## Phase 1: Data Ingestion & Anomaly Detection (Bot Filtering)

Before constructing the multi-touch attribution model, it is critical to ensure the integrity of the clickstream data. Raw marketing data is often polluted with non-human bot traffic, which can severely skew conversion metrics and Cost Per Acquisition (CPA) calculations.

In this pipeline stage, we load the raw datasets and apply two strict algorithmic thresholds to purge non-human interactions:
 **The Volume Threshold:** Identifies and flags any `User_ID` generating an impossibly high volume of actions (e.g., > 500 events).
 **The Speed Threshold:** Analyzes the chronological `Timestamp` of user actions. Any `User_ID` executing sequential actions in under 1.0 second is flagged as physically impossible for a human.

Finally, we isolate the verified human touchpoints and merge them with our `user_profiles` dataset to retain psychographic and demographic context for the journey mapping.

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the datasets from the data/ folder
print("Loading data...")
touchpoints_df = pd.read_csv('data/touchpoints.csv')
profiles_df = pd.read_csv('data/user_profiles.csv')
spend_df = pd.read_csv('data/campaign_spend.csv')

# Convert Timestamp to datetime to do time-math
if 'Timestamp' in touchpoints_df.columns:
    touchpoints_df['Timestamp'] = pd.to_datetime(touchpoints_df['Timestamp'])

print(f"Original touchpoints shape: {touchpoints_df.shape}")

# 2. Filter the Bot Traffic
# Logic A: Drop users with a massive, non-human volume of actions (e.g., > 500 actions)
actions_per_user = touchpoints_df.groupby('User_ID').size()
bot_users_volume = actions_per_user[actions_per_user > 500].index 

# Logic B: Impossibly fast click times (sub 1-second intervals between actions)
if 'Timestamp' in touchpoints_df.columns:
    touchpoints_df = touchpoints_df.sort_values(by=['User_ID', 'Timestamp'])
    touchpoints_df['time_diff'] = touchpoints_df.groupby('User_ID')['Timestamp'].diff().dt.total_seconds()
    bot_users_speed = touchpoints_df[touchpoints_df['time_diff'] < 1.0]['User_ID'].unique()
else:
    bot_users_speed = []

# Combine suspected bots and drop them
all_bots = set(bot_users_volume).union(set(bot_users_speed))
clean_touchpoints = touchpoints_df[~touchpoints_df['User_ID'].isin(all_bots)].copy()

if 'time_diff' in clean_touchpoints.columns:
    clean_touchpoints = clean_touchpoints.drop(columns=['time_diff'])

print(f"Cleaned touchpoints shape (Bots removed): {clean_touchpoints.shape}")
print(f"Total Bot IDs nuked: {len(all_bots)}")

# 3. Merge with User Profiles
# Safety check to ensure column names match before merging
if 'user_id' in profiles_df.columns:
    profiles_df = profiles_df.rename(columns={'user_id': 'User_ID'})

merged_df = pd.merge(clean_touchpoints, profiles_df, on='User_ID', how='left')

# 4. Print the final shape and verify
print(f"Final merged dataset shape: {merged_df.shape}")
merged_df.head()

Loading data...
Original touchpoints shape: (566510, 5)
Cleaned touchpoints shape (Bots removed): (318106, 5)
Total Bot IDs nuked: 11518
Final merged dataset shape: (318106, 8)


,User_ID,Timestamp,Campaign_ID,Channel,Event_Type,Segment,Trend_Affinity,Geography
0,U_B01_00000,2026-01-01 19:27:00,CMP_B01_INF_899,Influencer Blog,Impression,Fitness Enthusiast,Sustainable Packaging,Tier 1
1,U_B01_00001,2026-01-01 20:37:00,CMP_B01_GOO_434,Google Search,Impression,Fitness Enthusiast,Vegan,Tier 1
2,U_B01_00001,2026-01-01 20:38:00,CMP_B01_GOO_434,Google Search,Click,Fitness Enthusiast,Vegan,Tier 1
3,U_B01_00001,2026-01-04 15:37:00,CMP_B01_MAR_127,Marketplace,Impression,Fitness Enthusiast,Vegan,Tier 1
4,U_B01_00001,2026-01-06 08:37:00,CMP_B01_INS_285,Instagram,Impression,Fitness Enthusiast,Vegan,Tier 1


In [ ]:
print(touchpoints_df.columns)

Index(['User_ID', 'Timestamp', 'Campaign_ID', 'Channel', 'Event_Type',
       'time_diff'],
      dtype='str')


## Phase 2: Markov Chain Multi-Touch Attribution Engine

Historically, "Last-Click" attribution models inherently overvalue bottom-of-funnel channels (like Search) while starving top-of-funnel channels (like Social or Influencers) of credit. To map the true customer journey, this section constructs a probabilistic **Markov Chain Model**.

The pipeline executes the following mathematical sequence:
1. **Chronological Journey Mapping:** Aggregates user touchpoints into chronologically sorted arrays, mapping the exact path from `Start` to either `Conversion` or `Null` (abandonment).
2. **The Transition Matrix:** Calculates the exact state-to-state transition probabilities based on historical flow volume, creating a matrix of all possible customer movements.
3. **The Removal Effect (Simulation):** To determine the *true* weight of a channel, the algorithm mathematically deletes it from the matrix. By isolating the transient matrix ($Q$) and applying the fundamental matrix equation $N = (I - Q)^{-1}$, we simulate the new conversion probability as if the channel did not exist.
4. **Attribution Normalization:** The drop in conversion volume dictates the channel's actual importance. These "Removal Effects" are normalized into percentage-based True Attribution Weights.

In [3]:
import pandas as pd
import numpy as np
from collections import defaultdict

print("1. Mapping chronological user journeys...")
# Sort to ensure timestamps are perfectly chronological
merged_df = merged_df.sort_values(by=['User_ID', 'Timestamp'])

journeys = []
# Group by user to build their unique path
for user_id, group in merged_df.groupby('User_ID'):
    events = group['Event_Type'].values
    channels = group['Channel'].values
    
    journey = ['Start']
    converted = False
    
    for ch, ev in zip(channels, events):
        if ev == 'Purchase':
            converted = True
            break
        journey.append(ch)
        
    if converted:
        journey.append('Conversion')
    else:
        journey.append('Null')
        
    journeys.append(journey)

print(f"Total Journeys Mapped: {len(journeys)}")

print("2. Calculating the Markov Transition Matrix...")
# Count transitions from state A to state B
transitions = defaultdict(int)
state_counts = defaultdict(int)
states = set()

for journey in journeys:
    for i in range(len(journey) - 1):
        current_state = journey[i]
        next_state = journey[i + 1]
        transitions[(current_state, next_state)] += 1
        state_counts[current_state] += 1
        states.add(current_state)
        states.add(next_state)

states = list(states)
matrix_df = pd.DataFrame(0.0, index=states, columns=states)

# Convert raw counts to probabilities
for (state_from, state_to), count in transitions.items():
    matrix_df.loc[state_from, state_to] = count / state_counts[state_from]

# Lock the absorbing states (Once they convert or leave, they stay there)
if 'Conversion' in matrix_df.index:
    matrix_df.loc['Conversion', :] = 0.0
    matrix_df.loc['Conversion', 'Conversion'] = 1.0
if 'Null' in matrix_df.index:
    matrix_df.loc['Null', :] = 0.0
    matrix_df.loc['Null', 'Null'] = 1.0

print("3. Calculating True Removal Effects...")
total_journeys = len(journeys)
base_conversions = sum(1 for j in journeys if j[-1] == 'Conversion')
base_conversion_rate = base_conversions / total_journeys

removal_effects = {}
marketing_channels = [s for s in states if s not in ['Start', 'Conversion', 'Null']]

# Isolate transient and absorbing states for matrix inversion
transient_states = [s for s in states if s not in ['Conversion', 'Null']]
absorbing_states = ['Conversion', 'Null']
ordered_states = transient_states + absorbing_states
start_idx = transient_states.index('Start')
conv_idx = absorbing_states.index('Conversion')

for channel in marketing_channels:
    # Copy base matrix and "Remove" the channel by forcing it to drop to Null
    removal_matrix = matrix_df.copy()
    removal_matrix.loc[channel, :] = 0.0
    removal_matrix.loc[channel, 'Null'] = 1.0
    
    # Linear Algebra logic to calculate new conversion probability
    P = removal_matrix.loc[ordered_states, ordered_states].values
    t = len(transient_states)
    Q = P[:t, :t] # Transient to Transient
    R = P[:t, t:] # Transient to Absorbing
    
    # Fundamental matrix N = (I - Q)^-1
    try:
        I = np.eye(t)
        N = np.linalg.inv(I - Q)
        B = np.dot(N, R) # Absorption probabilities
        new_conv_prob = B[start_idx, conv_idx]
    except np.linalg.LinAlgError:
        new_conv_prob = 0.0
        
    # How much did conversion drop without this channel?
    removal_effect = (base_conversion_rate - new_conv_prob) / base_conversion_rate
    removal_effects[channel] = removal_effect

# 4. Normalize weights to equal 100%
total_effect = sum(removal_effects.values())
attribution_weights = {k: (v/total_effect)*100 for k, v in removal_effects.items()}

attribution_df = pd.DataFrame.from_dict(attribution_weights, orient='index', columns=['True_Attribution_Weight_%'])
attribution_df = attribution_df.sort_values(by='True_Attribution_Weight_%', ascending=False).round(2)

print("\n--- PHASE 1 COMPLETE: TRUE CHANNEL ATTRIBUTION ---")
print(attribution_df)

1. Mapping chronological user journeys...
Total Journeys Mapped: 88482
2. Calculating the Markov Transition Matrix...
3. Calculating True Removal Effects...

--- PHASE 1 COMPLETE: TRUE CHANNEL ATTRIBUTION ---
                 True_Attribution_Weight_%
Google Search                        21.83
Influencer Blog                      20.21
Instagram                            19.51
Marketplace                          19.37
YouTube                              19.08


In [5]:
print(spend_df.columns)

Index(['Campaign_ID', 'Brand_ID', 'Channel', 'Pricing_Model', 'Cost_Rate_INR',
       'Total_Budget_Allocated'],
      dtype='str')


## Phase 3: True CPA Calculation & Fatigue-Adjusted Budget Reallocation

With the True Attribution Weights established, we can reconcile the marketing performance against the historical financial ledger to calculate the actual Cost Per Acquisition (CPA) for each channel. 

To generate an actionable business strategy for the ₹10 Crore budget, this engine executes two final steps:
1. **True CPA Calculation:** Maps the historical financial spend to the Markov-derived conversion volumes, exposing highly inefficient channels (e.g., YouTube) and highly profitable ones (e.g., Influencer Blogs).
2. **Ad Fatigue Dampener (Square Root Transformation):** A naive allocation algorithm would dump 100% of the budget into the single channel with the lowest CPA. In reality, this rapidly triggers audience saturation and ad fatigue, causing CPA to skyrocket. To prevent this, we apply an inverse mathematical dampener (a square root transformation of the efficiency score). This safely scales funding to top-performing channels while maintaining a balanced, full-funnel marketing ecosystem.

In [8]:
import numpy as np

print("1. Calculating True CPA (Cost Per Acquisition)...")
# Ensure the budget column is numeric
spend_df['Total_Budget_Allocated'] = pd.to_numeric(spend_df['Total_Budget_Allocated'], errors='coerce')

# Get the total historical spend deployed per channel
historical_spend = spend_df.groupby('Channel')['Total_Budget_Allocated'].sum()

# Distribute the total historical conversions based on our Markov True Weights
channel_conversions = {}
for channel in attribution_df.index:
    weight = attribution_df.loc[channel, 'True_Attribution_Weight_%'] / 100
    # base_conversions carries over from the Markov cell
    channel_conversions[channel] = base_conversions * weight 

# Calculate True CPA
cpa_data = []
for channel in attribution_df.index:
    spend = historical_spend.get(channel, 0)
    conversions = channel_conversions.get(channel, 0)
    cpa = spend / conversions if conversions > 0 else 0
    cpa_data.append({
        'Channel': channel, 
        'Historical_Spend': spend, 
        'True_Conversions': conversions, 
        'True_CPA': cpa
    })

cpa_df = pd.DataFrame(cpa_data).set_index('Channel')
print(cpa_df[['Historical_Spend', 'True_CPA']].round(2))


print("\n2. Building the Fatigue-Adjusted Reallocation Engine...")
# Phase 2 requirement: ₹10 Crore per brand
BUDGET_PER_BRAND = 100_000_000 

# The Math: Lower CPA is better. We invert the CPA to get an Efficiency Score.
cpa_df['Efficiency_Score'] = 1 / cpa_df['True_CPA']

# Ad Fatigue Dampener: We apply a square root to the efficiency score. 
# This prevents the model from blindly dumping 100% of the budget into the #1 channel, 
# ensuring top-of-funnel channels still get the funding needed to prime the market.
cpa_df['Fatigue_Adjusted_Score'] = np.sqrt(cpa_df['Efficiency_Score'])

# Normalize the dampener scores to equal 100%
total_score = cpa_df['Fatigue_Adjusted_Score'].sum()
cpa_df['Optimized_Allocation_%'] = (cpa_df['Fatigue_Adjusted_Score'] / total_score) * 100

# Apply the percentage to the ₹10 Crore budget
cpa_df['Recommended_Budget_INR'] = (cpa_df['Optimized_Allocation_%'] / 100) * BUDGET_PER_BRAND

print("\n--- PHASE 2: FINAL STRATEGY FOR THE CMO (PER BRAND) ---")
# Format the output cleanly for the presentation
final_strategy = cpa_df[['True_CPA', 'Optimized_Allocation_%', 'Recommended_Budget_INR']].copy()
final_strategy['Recommended_Budget_INR'] = final_strategy['Recommended_Budget_INR'].apply(lambda x: f"₹ {x:,.0f}")

# Sort by the highest budget allocation
final_strategy = final_strategy.sort_values(by='Optimized_Allocation_%', ascending=False)

1. Calculating True CPA (Cost Per Acquisition)...
                 Historical_Spend   True_CPA
Channel                                     
Google Search        2.029821e+08  189220.79
Influencer Blog      1.148977e+08  115693.75
Instagram            1.941355e+08  202494.19
Marketplace          2.405746e+08  252746.44
YouTube              2.660160e+08  283722.80

2. Building the Fatigue-Adjusted Reallocation Engine...

--- PHASE 2: FINAL STRATEGY FOR THE CMO (PER BRAND) ---
